# Agent 4 — KPI Baseline Computation

Computes historical baseline statistics for all registered KPIs.
Uses fixed pilot data dates rather than relative date offsets.

Run this before running the KPI validation agent.
Safe to re-run — uses MERGE to avoid duplicates.

Reads from:
  uc4_kpi_registry_b
  uc2_gold_daily_store_metrics
  uc2_gold_daily_inventory_summary

Writes to:
  uc4_kpi_baseline_metrics_b

## Cell 2: Imports and Configuration

In [0]:
import uuid
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, TimestampType,
    DoubleType, IntegerType, BooleanType
)
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Table References ───────────────────────────────────────────────────────

KPI_REGISTRY         = f"{CATALOG}.{SCHEMA}.uc4_kpi_registry_b"
KPI_BASELINE_METRICS = f"{CATALOG}.{SCHEMA}.uc4_kpi_baseline_metrics_b"

# ── Fixed Date Configuration ───────────────────────────────────────────────
# Hardcoded to match actual pilot data dates.
# Running this notebook on any future date always computes
# baselines from the same clean historical window.
#
# Pilot data coverage:
#   Historical clean : 2026-05-19 to 2026-06-17
#   Clean live batch : 2026-06-19
#   Controlled batches (EXCLUDED): 2026-06-20 to 2026-06-24

BASELINE_COMPUTATION_DATE = date(2026, 6, 25)   # logical computation date

# Baseline window covers all clean historical dates
# plus the clean live batch 2026-06-19
BASELINE_WINDOW_START = date(2026, 5, 19)
BASELINE_WINDOW_END   = date(2026, 6, 19)

# Dates explicitly excluded from baseline calculation
# These have injected anomalies that would corrupt the mean
EXCLUDE_DATES = [
    "2026-06-18",   # no data exists for this date
    "2026-06-20",   # Agent 1 volume drop injected
    "2026-06-21",   # Agent 1 null spike injected
    "2026-06-22",   # Agent 4 Scenario A: impossible return_rate
    "2026-06-23",   # Agent 4 Scenario B: basket value 60% drop
    "2026-06-24",   # Agent 4 Scenario C: formula mismatch
]

BASELINE_WINDOW_DAYS = (
    (BASELINE_WINDOW_END - BASELINE_WINDOW_START).days + 1
)
MIN_SAMPLE_COUNT = 7
COMPUTED_BY      = "01_kpi_baseline_computation"

# ── Baseline Schema ────────────────────────────────────────────────────────

BASELINE_SCHEMA = StructType([
    StructField("baseline_id",               StringType(),    False),
    StructField("gold_fully_qualified_name",  StringType(),    False),
    StructField("kpi_name",                  StringType(),    False),
    StructField("kpi_column",                StringType(),    False),
    StructField("baseline_date",             DateType(),      False),
    StructField("baseline_window_days",      IntegerType(),   False),
    StructField("mean_value",                DoubleType(),    True),
    StructField("stddev_value",              DoubleType(),    True),
    StructField("min_value",                 DoubleType(),    True),
    StructField("max_value",                 DoubleType(),    True),
    StructField("p25_value",                 DoubleType(),    True),
    StructField("p50_value",                 DoubleType(),    True),
    StructField("p75_value",                 DoubleType(),    True),
    StructField("p95_value",                 DoubleType(),    True),
    StructField("sample_count",              IntegerType(),   False),
    StructField("is_sufficient_data",        BooleanType(),   False),
    StructField("baseline_status",           StringType(),    False),
    StructField("computed_timestamp",        TimestampType(), False),
    StructField("computed_by",               StringType(),    False),
])

print("=" * 60)
print("AGENT 4 — KPI BASELINE COMPUTATION")
print("=" * 60)
print(f"Computation date  : {BASELINE_COMPUTATION_DATE}")
print(f"Baseline start    : {BASELINE_WINDOW_START}")
print(f"Baseline end      : {BASELINE_WINDOW_END}")
print(f"Window days       : {BASELINE_WINDOW_DAYS}")
print(f"Min sample count  : {MIN_SAMPLE_COUNT}")
print()
print(f"Excluded dates ({len(EXCLUDE_DATES)}):")
for d in EXCLUDE_DATES:
    print(f"  {d}")
print("=" * 60)

## Cell 3: Load KPI Registry

In [0]:
kpis = (
    spark.table(KPI_REGISTRY)
         .filter(F.col("is_active") == True)
         .collect()
)

print(f"Active KPIs registered: {len(kpis)}")
for kpi in kpis:
    checks = kpi["checks_enabled"] or "[]"
    print(
        f"  {kpi['kpi_id']:<20} "
        f"{kpi['kpi_name']:<35} "
        f"checks={checks}"
    )

## Cell 4: Compute Baselines

In [0]:
all_baseline_records = []

print(f"Computing baselines for {len(kpis)} KPIs...")
print(
    f"Date filter: {BASELINE_WINDOW_START} to {BASELINE_WINDOW_END} "
    f"excluding {len(EXCLUDE_DATES)} dates"
)
print()

for kpi in kpis:
    kpi_id      = kpi["kpi_id"]
    gold_fqn    = kpi["gold_fully_qualified_name"]
    kpi_name    = kpi["kpi_name"]
    kpi_column  = kpi["kpi_column"]
    biz_date_col= kpi["business_date_column"]
    checks      = kpi["checks_enabled"] or "[]"

    # Only compute baselines for KPIs with ANOMALY checks
    # FORMULA and CROSS_KPI checks do not need historical baselines
    if "ANOMALY" not in checks:
        print(f"  Skipping {kpi_name} (no ANOMALY check configured)")
        continue

    print(
        f"  Computing baseline for {kpi_name} "
        f"in {gold_fqn.split('.')[-1]}"
    )

    try:
        df = spark.table(gold_fqn)

        if kpi_column not in df.columns:
            print(f"    Column {kpi_column} not found in table. Skipping.")
            continue

        # Apply fixed date filter excluding injected anomaly dates
        if biz_date_col and biz_date_col in df.columns:
            df_filtered = df.filter(
                (F.col(biz_date_col) >= F.lit(str(BASELINE_WINDOW_START))) &
                (F.col(biz_date_col) <= F.lit(str(BASELINE_WINDOW_END))) &
                (~F.col(biz_date_col).isin(EXCLUDE_DATES))
            )
        else:
            df_filtered = df

        filtered_count = df_filtered.count()
        print(f"    Records in baseline window: {filtered_count}")

        if filtered_count == 0:
            print(f"    No records in baseline window. Skipping.")
            continue

        # Compute statistics
        stats = df_filtered \
            .filter(F.col(kpi_column).isNotNull()) \
            .agg(
                F.mean(kpi_column)
                 .alias("mean_value"),
                F.stddev(kpi_column)
                 .alias("stddev_value"),
                F.min(kpi_column)
                 .alias("min_value"),
                F.max(kpi_column)
                 .alias("max_value"),
                F.percentile_approx(kpi_column, 0.25)
                 .alias("p25_value"),
                F.percentile_approx(kpi_column, 0.50)
                 .alias("p50_value"),
                F.percentile_approx(kpi_column, 0.75)
                 .alias("p75_value"),
                F.percentile_approx(kpi_column, 0.95)
                 .alias("p95_value"),
                F.count("*")
                 .alias("sample_count")
            ).collect()[0]

        sample_count  = int(stats["sample_count"] or 0)
        is_sufficient = sample_count >= MIN_SAMPLE_COUNT

        record = {
            "baseline_id"              : str(uuid.uuid4()),
            "gold_fully_qualified_name": gold_fqn,
            "kpi_name"                 : kpi_name,
            "kpi_column"               : kpi_column,
            "baseline_date"            : BASELINE_COMPUTATION_DATE,
            "baseline_window_days"     : BASELINE_WINDOW_DAYS,
            "mean_value"               : float(stats["mean_value"] or 0),
            "stddev_value"             : float(stats["stddev_value"] or 0),
            "min_value"                : float(stats["min_value"] or 0),
            "max_value"                : float(stats["max_value"] or 0),
            "p25_value"                : float(stats["p25_value"] or 0),
            "p50_value"                : float(stats["p50_value"] or 0),
            "p75_value"                : float(stats["p75_value"] or 0),
            "p95_value"                : float(stats["p95_value"] or 0),
            "sample_count"             : sample_count,
            "is_sufficient_data"       : is_sufficient,
            "baseline_status"          : "VALID" if is_sufficient else "INSUFFICIENT_DATA",
            "computed_timestamp"       : datetime.now(),
            "computed_by"              : COMPUTED_BY
        }

        all_baseline_records.append(record)

        print(
            f"    Mean: {record['mean_value']:.4f}  "
            f"StdDev: {record['stddev_value']:.4f}  "
            f"Samples: {sample_count}  "
            f"Status: {record['baseline_status']}"
        )

    except Exception as e:
        logger.error(f"Failed to compute baseline for {kpi_name}: {str(e)}")
        print(f"    ERROR: {str(e)}")

print(f"\nBaseline records computed: {len(all_baseline_records)}")

## Cell 5: Write Baselines

In [0]:
if not all_baseline_records:
    print("No baseline records to write.")
else:
    col_order = [f.name for f in BASELINE_SCHEMA.fields]
    rows      = [
        tuple(r.get(col) for col in col_order)
        for r in all_baseline_records
    ]
    df = spark.createDataFrame(rows, schema=BASELINE_SCHEMA)

    df.createOrReplaceTempView("kpi_baseline_staging")

    spark.sql(f"""
        MERGE INTO {KPI_BASELINE_METRICS} AS target
        USING kpi_baseline_staging AS source
        ON  target.gold_fully_qualified_name = source.gold_fully_qualified_name
        AND target.kpi_name                  = source.kpi_name
        AND target.kpi_column                = source.kpi_column
        AND target.baseline_date             = source.baseline_date
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)

    print(f"Written {len(all_baseline_records)} baseline records.")

## Cell 6: Verify Baselines

In [0]:
print("=" * 60)
print("KPI BASELINE SUMMARY")
print("=" * 60)

spark.sql(f"""
    SELECT
        SUBSTRING_INDEX(
            gold_fully_qualified_name, '.', -1
        )                           AS gold_table,
        kpi_name,
        kpi_column,
        ROUND(mean_value, 4)        AS mean_value,
        ROUND(stddev_value, 4)      AS stddev_value,
        ROUND(min_value, 4)         AS min_value,
        ROUND(max_value, 4)         AS max_value,
        sample_count,
        baseline_status
    FROM {KPI_BASELINE_METRICS}
    WHERE baseline_date = '{BASELINE_COMPUTATION_DATE}'
    ORDER BY gold_fully_qualified_name, kpi_name
""").show(20, truncate=False)

## Cell 7: Final Summary

In [0]:
print("\n" + "=" * 60)
print("KPI BASELINE COMPUTATION COMPLETE")
print("=" * 60)
print(f"Computation date  : {BASELINE_COMPUTATION_DATE}")
print(f"Baseline window   : {BASELINE_WINDOW_START} to {BASELINE_WINDOW_END}")
print(f"Excluded dates    : {EXCLUDE_DATES}")
print(f"Records computed  : {len(all_baseline_records)}")
print(f"Completed at      : {datetime.now()}")
print("=" * 60)

if all_baseline_records:
    print("\nBaselines are ready.")
    print("Next: Run 02_kpi_validation_agent")
else:
    print("\nWARNING: No baselines computed.")
    print("Check that Gold tables have data in the baseline window.")